<a href="https://colab.research.google.com/github/sabithakrishnan/RAG-medical-assistant/blob/main/performance_measures_of_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get update
!sudo apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Install libraries
!pip install -q langchain langchain-community langchain-huggingface chromadb pypdf sentence-transformers

# Install and Start Ollama
!curl -fsSL https://ollama.com | sh
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(10)
!ollama pull cniongolo/biomistral


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
import os
import subprocess # Import subprocess
import time       # Import time
import requests   # For checking Ollama server status

# --- 1. INITIALIZE RAG ---
def initialize_rag():
    # Make sure to mount drive first if your file is there!
    # Or change this path to a file you uploaded to Colab
    pdf_path = "/content/drive/MyDrive/thyroid-patient.pdf"

    if not os.path.exists(pdf_path):
        print(f"Error: Could not find file at {pdf_path}")
        return None

    # --- Start Ollama server if not running ---
    ollama_api_url = "http://localhost:11434"
    try:
        # Try to make a simple request to check if Ollama is running
        requests.get(f"{ollama_api_url}/api/tags", timeout=1)
        print("Ollama server is already running.")
    except requests.exceptions.ConnectionError:
        print("Starting Ollama server...")
        # Use nohup to run ollama serve in the background and detach it from the current shell
        # Redirect output to a log file to avoid terminal clutter
        subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
        time.sleep(10) # Give it some time to start

        # Verify Ollama server is up
        start_time = time.time()
        while True:
            try:
                requests.get(f"{ollama_api_url}/api/tags", timeout=1)
                print("Ollama server started successfully.")
                break
            except requests.exceptions.ConnectionError:
                if time.time() - start_time > 60: # Timeout after 60 seconds
                    print("Error: Ollama server failed to start after 60 seconds.")
                    return None
                time.sleep(5) # Wait a bit before retrying
    except requests.exceptions.Timeout:
        print("Ollama server responded but timed out, proceeding.")
    except Exception as e:
        print(f"An unexpected error occurred while checking Ollama server: {e}")
        return None

    # --- Ensure the model is pulled ---
    try:
        # Check if the model is already available by trying to list local models
        response = requests.get(f"{ollama_api_url}/api/tags")
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        models_data = response.json()
        # The model name in `ollama list` output might include ':latest' or a digest
        # Let's check if any model starts with 'cniongolo/biomistral'
        model_found = False
        for model_info in models_data.get('models', []):
            if model_info['name'].startswith('cniongolo/biomistral'):
                model_found = True
                break

        if not model_found:
            print("Ollama model 'cniongolo/biomistral' not found locally. Pulling it now...")
            subprocess.run(["ollama", "pull", "cniongolo/biomistral"], check=True)
            print("Ollama model 'cniongolo/biomistral' pulled successfully.")
        else:
            print("Ollama model 'cniongolo/biomistral' is already available.")
    except requests.exceptions.RequestException as e:
        print(f"Error checking/pulling Ollama model: {e}")
        return None
    except subprocess.CalledProcessError as e:
        print(f"Error pulling Ollama model: {e}")
        return None

    loader = PyPDFLoader(pdf_path)
    data = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
    chunks = text_splitter.split_documents(data)

    embeddings = HuggingFaceEmbeddings(model_name="dmis-lab/biobert-v1.1")
    vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings)

    llm = Ollama(model="cniongolo/biomistral")

    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        memory=memory
    )
    return chain

# --- 2. START THE INTERACTIVE CHAT ---
print("🩺Initializing Medical AI... (This may take a minute)")
qa_chain = initialize_rag()

if qa_chain:
    print("\n" + "="*50)
    print(" THYROID MEDICAL ASSISTANT READY")
    print("Type your questions below. Type 'exit' to quit.")
    print("="*50 + "\n")

    while True:
        query = input("User: ")

        if query.lower() in ['exit', 'quit', 'bye']:
            print("Assistant: Goodbye! Stay healthy.")
            break

        print("Assistant: (Thinking...)", end="\r")
        result = qa_chain.invoke({"question": query})
        print(f"Assistant: {result['answer']}\n")

In [ ]:
medical_eval_set = [
    {
        "question": "What are the two main hormones produced by the thyroid and what do they regulate?",
        "ground_truth": "The thyroid produces thyroxine (T4) and triiodothyronine (T3). They help regulate body temperature, blood pressure, heart rate, weight, and metabolism."
    },
    {
        "question": "Who is at the highest risk for thyroid cancer based on age and sex?",
        "ground_truth": "Those assigned female at birth are 3 times more likely to be diagnosed. It is the most common cancer in adults aged 18 to 33 years."
    },
    {
        "question": "What is the purpose of a TSH blood test if it cannot diagnose thyroid cancer?",
        "ground_truth": "The level is checked to see if the nodule is producing thyroid hormone. Nodules that produce thyroid hormone are rarely cancerous."
    },
    {
        "question": "What are the long-term side effects of a total thyroidectomy?",
        "ground_truth": "Long-term side effects can include hypoparathyroidism (low calcium levels) and damage to the nerves controlling the voice and swallowing."
    },
    {
        "question": "Against which types of thyroid cancer is Radioactive Iodine (RAI) therapy NOT effective?",
        "ground_truth": "RAI therapy is not effective against medullary or anaplastic thyroid cancer."
    },
    {
        "question": "What gene mutation is responsible for approximately 1 in 4 medullary thyroid cancer cases?",
        "ground_truth": "About 1 in 4 medullary thyroid cancers is caused by a pathogenic variant (mutation) in the RET gene."
    },
    {
        "question": "How is Anaplastic Thyroid Cancer (ATC) staged and what is its typical prognosis?",
        "ground_truth": "All anaplastic thyroid cancers are considered stage IV (4). It is the most aggressive type and generally cannot be cured."
    }
]


In [ ]:
!pip install RAGAS

In [ ]:
import time
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

def run_performance_evaluation(qa_chain, eval_set):
    results_data = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": [],
        "latency": []
    }

    print("📊 Running Performance Evaluation...")

    for item in eval_set:
        start_time = time.time()

        # Invoke the chain
        response = qa_chain.invoke({"question": item['question']})

        latency = time.time() - start_time

        results_data["question"].append(item['question'])
        results_data["answer"].append(response['answer'])
        # Extract the text from retrieved documents
        results_data["contexts"].append([doc.page_content for doc in response['source_documents']])
        results_data["ground_truth"].append(item['ground_truth'])
        results_data["latency"].append(latency)

    # Convert to Dataset for RAGAS
    dataset = Dataset.from_dict(results_data)

    # Calculate RAGAS Metrics
    # Note: RAGAS usually requires an OpenAI API key for LLM-as-a-judge,
    # but you can configure it to use your Mistral model.
    score = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
    )

    # Print Summary
    print("\n--- PERFORMANCE SUMMARY ---")
    print(f"Average Latency: {sum(results_data['latency'])/len(results_data['latency']):.2f} seconds")
    print(score)

    return score

# To execute:
eval_scores = run_performance_evaluation(qa_chain, medical_eval_set)
